# Previsão diária de acidentes rodoviários por tipo em Portugal continental

**Objetivo.** Prever separadamente o número diário de despistes, colisões e atropelamentos nos 18 distritos de Portugal continental.

1. Com que erro consegue um modelo de machine learning prever a contagem diária de cada tipo em anos fora do treino?
2. Para que tipos melhora face a uma referência histórica baseada no mês e no dia da semana?
3. Em que medida a meteorologia e a luz natural acrescentam capacidade preditiva?

Este notebook lê a base diária preparada e descreve as distribuições, os padrões temporais e as associações com a meteorologia. A comparação de previsões entre SARIMA e redes neuronais pertence à fase de modelação. O teste de 2024 fica reservado para essa comparação.

**Unidade e horizonte.** Uma linha por data civil, agregada para Portugal continental. O dia é o índice e origina variáveis de calendário. As três respostas são as contagens do dia-alvo. O histórico usa datas anteriores e a disponibilidade atempada dos registos tem de ser confirmada. A meteorologia ERA5 do próprio dia é reanálise e permite uma avaliação retrospetiva. Para uma previsão operacional, esse bloco requer previsões meteorológicas arquivadas que estivessem disponíveis na hora de emissão.

**Dados.** A entrada é `analise/dados/serie_diaria_continente.csv`. O notebook `preparar_serie_diaria.ipynb` documenta e executa o filtro de Portugal continental e a agregação da meteorologia. O guia `analise/README.md` descreve o formato da base.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

atual = Path.cwd()
caminho_relativo = Path("dados/serie_diaria_continente.csv")
caminho = next(
    (p / caminho_relativo for p in (atual, atual / "analise", *atual.parents)
     if (p / caminho_relativo).is_file()),
    None,
)
if caminho is None:
    raise FileNotFoundError("Executar preparar_serie_diaria.ipynb para gerar a base diária.")

TIPOS = ["Despiste", "Colisão", "Atropelamento"]
DIAS_SEMANA = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
CORES = dict(zip(TIPOS, ["#00747A", "#3459A6", "#B44429"]))
diario = pd.read_csv(
    caminho, parse_dates=["data"], index_col="data", dtype={tipo: "Int64" for tipo in TIPOS},
)
calendario = pd.date_range("2010-01-01", "2024-12-31", freq="D", name="data")
meteorologicas = diario.columns.difference(["periodo", *TIPOS], sort=False).tolist()
assert diario.index.equals(calendario)
assert diario[TIPOS].notna().all(axis=1).eq(diario["periodo"].ne("sem_dados")).all()
assert diario[meteorologicas].notna().all().all()
plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False, "axes.spines.right": False})

## Distribuições e escolha inicial das entradas

As tabelas e os gráficos desta secção usam 2010–2019. A assimetria e os dias sem ocorrências motivam modelos de contagem. A dispersão bruta mistura estações e dias da semana, pelo que a verificação de Poisson usa também a dispersão **após o ajuste**.

| Entrada candidata | Justificação e decisão a testar |
|---|---|
| Dia da semana | Captura diferenças semanais na circulação. Usar como categoria com segunda-feira como referência no GLM. |
| Época do ano | Seno e cosseno representam sazonalidade anual contínua. A referência histórica usa o mês. |
| Tendência temporal | Testar se a evolução de longo prazo melhora a validação. Avaliar com atenção em 2023, após a lacuna. |
| Contagem D−2 e média D−8 a D−2 | Captam a atividade recente de cada tipo. Comparar isoladamente e em conjunto. Confirmar a disponibilidade na origem da previsão. |
| Precipitação total | A chuva pode alterar as condições de circulação. Descrever os zeros e usar log(1 + mm) para reduzir a influência dos valores extremos. |
| Extensão geográfica da chuva | Uma média baixa no continente pode resultar de chuva intensa localizada. Testar a extensão como alternativa e como complemento ao total. |
| Temperatura média e vento máximo | Podem captar condições meteorológicas distintas. Testar ganho por variável e em bloco. |
| Nebulosidade, duração de sol direto e radiação | Descrevem fenómenos relacionados. Comparar como alternativas e inspecionar a redundância antes de escolher. |
| Duração astronómica do dia | Conhecida à partida. Testar se traz informação adicional à sazonalidade anual. |

`Natureza` é usada para obter as respostas. Variáveis do acidente do próprio dia, como os veículos, lesões e luminosidade registada, só são conhecidas depois da ocorrência. O calendário dos jogos do SC Braga tem um âmbito local e fica fora desta análise de Portugal continental.

In [ ]:
diario["dia_semana"] = diario.index.day_of_week
diario["mes"] = diario.index.month
diario["tendencia_anos"] = (diario.index - calendario[0]).days / 365.25
fase = 2 * np.pi * (diario.index.day_of_year - 1) / np.where(diario.index.is_leap_year, 366, 365)
diario["seno_anual"] = np.sin(fase)
diario["cosseno_anual"] = np.cos(fase)
diario["log_precipitacao"] = np.log1p(diario["precipitacao_total_mm"])

for tipo in TIPOS:
    anterior = diario[tipo].astype(float).shift(2)
    diario[f"{tipo}_d2"] = anterior
    diario[f"{tipo}_media7"] = anterior.rolling(7, min_periods=7).mean()

assert diario.loc["2020":"2022", TIPOS].isna().all().all()
assert diario.loc["2023-01-01":"2023-01-08", [f"{t}_media7" for t in TIPOS]].isna().all().all()
assert diario.loc["2023-01-09", [f"{t}_media7" for t in TIPOS]].notna().all()

treino = diario.loc[diario["periodo"].eq("desenvolvimento")]
alvos = treino[TIPOS].astype(float)
resumo = alvos.describe(percentiles=[.05, .25, .5, .75, .95]).T
resumo["variancia"] = alvos.var()
resumo["variancia/media"] = alvos.var() / alvos.mean()
resumo["dias_zero_pct"] = alvos.eq(0).mean() * 100
display(resumo.round(2))
display(alvos.groupby(alvos.index.year).mean().round(2).rename_axis("Ano · média por dia"))

colunas_eda = meteorologicas + [f"{t}_{h}" for t in TIPOS for h in ("d2", "media7")]
resumo_x = treino[colunas_eda].describe(percentiles=[.05, .25, .5, .75, .95]).T
resumo_x["omissos"] = treino[colunas_eda].isna().sum()
resumo_x["dias_zero_pct"] = treino[colunas_eda].eq(0).mean() * 100
display(resumo_x.round(2))

fig, ax = plt.subplots(3, 2, figsize=(12, 8), layout="constrained")
for i, tipo in enumerate(TIPOS):
    y = alvos[tipo]
    ax[i, 0].hist(y, bins=np.arange(y.max() + 2) - .5, color=CORES[tipo])
    ax[i, 0].set(title=f"{tipo} · distribuição diária", xlabel="Acidentes por dia", ylabel="Dias")
    ax[i, 1].plot(y.rolling(28).mean(), color=CORES[tipo])
    ax[i, 1].set(title=f"{tipo} · média móvel de 28 dias", ylabel="Acidentes por dia")
plt.show()

fig, ax = plt.subplots(3, 2, figsize=(11, 7), layout="constrained")
for i, tipo in enumerate(TIPOS):
    y = alvos[tipo]
    ax[i, 0].bar(DIAS_SEMANA, y.groupby(y.index.day_of_week).mean(), color=CORES[tipo])
    ax[i, 1].plot(range(1, 13), y.groupby(y.index.month).mean(), marker="o", color=CORES[tipo])
    ax[i, 1].set(xticks=range(1, 13), xlabel="Mês")
    ax[i, 0].set(ylabel=f"{tipo}\nMédia por dia")
ax[0, 0].set_title("Perfil semanal")
ax[0, 1].set_title("Perfil mensal")
plt.show()

In [ ]:
nomes = {
    "temperatura_media_c": "Temperatura (°C)", "precipitacao_total_mm": "Precipitação (mm)",
    "vento_max_kmh": "Vento (km/h)", "nebulosidade_media_pct": "Nebulosidade (%)",
    "duracao_dia_h": "Duração do dia (h)", "sol_direto_h": "Sol direto (h)",
    "radiacao_total_mj_m2": "Radiação (MJ/m²)",
    "areas_com_chuva_pct": "Extensão da chuva (% ponderada)",
}
fig, ax = plt.subplots(2, 4, figsize=(13, 5.5), layout="constrained")
for eixo, (coluna, nome) in zip(ax.flat, nomes.items()):
    eixo.hist(treino[coluna], bins=30, color="#287D80")
    eixo.set(title=nome, ylabel="Dias")
plt.show()

corr = treino[[*nomes, "seno_anual", "cosseno_anual"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9.5, 6.3), layout="constrained")
cores = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
etiquetas = [nomes.get(c, c.replace("_", " ")) for c in corr]
ax.set_xticks(range(len(corr)), etiquetas, rotation=42, ha="right")
ax.set_yticks(range(len(corr)), etiquetas)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if abs(v) > .68 else "#222")
ax.set_title("Spearman entre meteorologia, luz e sazonalidade")
fig.colorbar(cores, ax=ax, shrink=.8)
plt.show()

quartis = []
for coluna in nomes:
    classe = pd.qcut(treino[coluna], 4, duplicates="drop")
    quadro = treino.groupby(classe, observed=True)[TIPOS].mean().reset_index(drop=True)
    quadro.insert(0, "Grupo (baixo → alto)", range(1, len(quadro) + 1))
    quadro.insert(0, "Variável", nomes[coluna])
    quartis.append(quadro)
display(pd.concat(quartis, ignore_index=True).round(2))

### Evolução anual e composição dos acidentes

As médias anuais usam os dias observados de cada ano. O painel da esquerda mostra
a evolução da frequência, enquanto o da direita mostra a proporção de cada tipo
no total de acidentes. Os dois painéis usam os anos de desenvolvimento (2010–2019).

In [ ]:
media_por_ano = alvos.groupby(alvos.index.year).mean()
total_por_ano = alvos.groupby(alvos.index.year).sum()
composicao_por_ano = 100 * total_por_ano.div(total_por_ano.sum(axis=1), axis=0)

fig, (ax_media, ax_composicao) = plt.subplots(1, 2, figsize=(12, 4.3), layout="constrained")
for tipo in TIPOS:
    ax_media.plot(media_por_ano.index, media_por_ano[tipo], marker="o", color=CORES[tipo], label=tipo)
ax_media.set(title="Contagem média diária por ano", ylabel="Acidentes por dia",
             xticks=media_por_ano.index)

ax_composicao.stackplot(
    composicao_por_ano.index, *(composicao_por_ano[tipo] for tipo in TIPOS),
    colors=[CORES[tipo] for tipo in TIPOS], alpha=.92,
)
ax_composicao.set(title="Composição anual", ylabel="Percentagem de acidentes",
                  xticks=composicao_por_ano.index, ylim=(0, 100))
for eixo in (ax_media, ax_composicao):
    eixo.set_xlabel("Ano")
    eixo.tick_params(axis="x", labelrotation=45)
    eixo.grid(axis="y", alpha=.2)
    eixo.set_axisbelow(True)
fig.legend(*ax_media.get_legend_handles_labels(), loc="outside lower center", ncols=3, frameon=False)
plt.show()

### Efeito conjunto do mês e do dia da semana

Cada célula indica quanto a média observada para aquela combinação fica acima
ou abaixo da média diária do próprio tipo de acidente no período de análise.
A escala é comum aos três painéis. Um valor de +20% corresponde a uma média
20% acima da média diária daquele tipo.

In [ ]:
from matplotlib.ticker import PercentFormatter

meses_curtos = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
                "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
medias_calendario = treino.groupby(["dia_semana", "mes"])[TIPOS].mean()
desvios_calendario = 100 * (medias_calendario.div(alvos.mean(), axis=1) - 1)
limite_cor = np.ceil(desvios_calendario.abs().to_numpy(dtype=float).max() / 10) * 10

fig, eixos = plt.subplots(3, 1, figsize=(10, 7.6), layout="constrained")
for eixo, tipo in zip(eixos, TIPOS, strict=True):
    desvio = desvios_calendario[tipo].unstack("mes")
    imagem = eixo.imshow(desvio.to_numpy(dtype=float), cmap="RdBu_r",
                         vmin=-limite_cor, vmax=limite_cor, aspect="auto")
    eixo.set(title=tipo)
    eixo.set_xticks(range(12), meses_curtos)
    eixo.set_yticks(range(7), DIAS_SEMANA)
    for i, dia in enumerate(desvio.index):
        for j, mes in enumerate(desvio.columns):
            valor = desvio.loc[dia, mes]
            eixo.text(j, i, f"{valor:+.0f}", ha="center", va="center", fontsize=7.5,
                      color="white" if abs(valor) > .6 * limite_cor else "#222")
barra = fig.colorbar(imagem, ax=eixos, shrink=.85, format=PercentFormatter(xmax=100, decimals=0))
barra.set_label("Desvio face à média diária do tipo")
plt.show()

### Contagens por condições meteorológicas

Os grupos de chuva separam os dias sem precipitação dos intervalos até 1 mm,
de 1 a 5 mm e acima de 5 mm. Os valores referem-se à precipitação média
ponderada no continente. Temperatura e radiação são divididas em quatro
grupos do mais baixo ao mais alto. O número abaixo de cada grupo indica
quantos dias contribuem para as médias.

Estas comparações descrevem os dados. Diferenças no calendário e na
circulação podem ajudar a explicar as diferenças entre grupos.

In [ ]:
grupos_meteo = {
    "Precipitação média (mm)": pd.cut(
        treino["precipitacao_total_mm"], bins=[-np.inf, 0, 1, 5, np.inf],
        labels=["0", ">0 a 1", ">1 a 5", ">5"],
    ),
    "Temperatura média (°C)": pd.qcut(
        treino["temperatura_media_c"], 4, labels=["Q1", "Q2", "Q3", "Q4"],
    ),
    "Radiação (MJ/m²)": pd.qcut(
        treino["radiacao_total_mj_m2"], 4, labels=["Q1", "Q2", "Q3", "Q4"],
    ),
}
fig, eixos = plt.subplots(1, 3, figsize=(12, 4.4), sharey=True, layout="constrained")
for eixo, (titulo, grupos) in zip(eixos, grupos_meteo.items(), strict=True):
    por_grupo = alvos.groupby(grupos, observed=True)
    medias = por_grupo.mean()
    n_dias = por_grupo.size()
    posicoes = np.arange(len(medias))
    for tipo in TIPOS:
        eixo.plot(posicoes, medias[tipo], color=CORES[tipo], marker="o",
                  linestyle="none", label=tipo)
    eixo.set(title=titulo)
    eixo.set_xticks(posicoes, [f"{grupo}\n(n={n_dias.loc[grupo]})" for grupo in medias.index])
    eixo.grid(axis="y", alpha=.25)
    eixo.set_axisbelow(True)
eixos[0].set_ylabel("Média de acidentes por dia")
fig.legend(*eixos[0].get_legend_handles_labels(), loc="outside lower center", ncols=3, frameon=False)
plt.show()

### Relação com os dias anteriores

A autocorrelação compara a contagem de um tipo com a contagem do mesmo tipo
alguns dias antes. As linhas verticais marcam semanas completas. O gráfico
usa a série diária de 2010–2019, pelo que inclui os padrões de calendário
e a evolução ao longo do período.

In [ ]:
atrasos = np.arange(1, 29)
fig, eixos = plt.subplots(1, 3, figsize=(12, 3.7), sharey=True, layout="constrained")
for eixo, tipo in zip(eixos, TIPOS, strict=True):
    for semana_completa in (7, 14, 21, 28):
        eixo.axvline(semana_completa, color="#b3b3b3", lw=.8, ls="--", zorder=0)
    autocorr = [alvos[tipo].autocorr(int(atraso)) for atraso in atrasos]
    eixo.bar(atrasos, autocorr, width=.72, color=CORES[tipo])
    eixo.axhline(0, color="#555", linewidth=.8)
    eixo.set(title=tipo, xlabel="Atraso em dias", xticks=[1, 7, 14, 21, 28])
eixos[0].set_ylabel("Autocorrelação das contagens")
plt.show()

### Resultados para a fase de modelação

- O padrão semanal muda com o tipo. Os despistes têm médias mais altas ao fim de semana, enquanto colisões e atropelamentos têm médias mais altas à sexta-feira.
- As colisões desceram de 2010 até 2012 e recuperaram até 2019. A evolução anual ajuda a escolher o tratamento da tendência na modelação.
- A precipitação média e a extensão da chuva têm correlação de Spearman de cerca de 0,94. A radiação e a duração de sol direto têm cerca de 0,97. São pares a comparar como entradas alternativas.
- Para testar a contribuição da meteorologia, comparar previsões com e sem essas variáveis nos mesmos dias e usar informação disponível no momento previsto de emissão. Com regressores externos, a extensão de SARIMA chama-se SARIMAX.